In [1]:
# !pip install grad-cam

In [2]:
# !pip install numpy-hilbert-curve

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from PIL import Image

import torch
import torch.nn.functional as F
import torch.nn as nn
from torchvision.transforms import v2
import torchvision.transforms as transforms
from torchvision.models import vit_l_16, ViT_L_16_Weights

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.base_cam import BaseCAM

from scripts.model_scripts import Custom_ResNet50, Custom_DenseNet121, Custom_ResNet101, ModelWrapper
from scripts.preprocessing_scripts import get_hilbert_index, hilbert_ravel


# Grad-CAM Analysis

In [14]:
### Set parameters
"""
model_type: The model architecture parameter ("ResNet50" or "DenseNet121")
dim: The input image dimension parameter (224, 384, or 512)
enhanced: Use models trained on enhanced images (True or False
row_idxs: The indexes to image rows from the input .csv
    (If this parameter is empty, no grad-CAMs are printed and saved)

csv_path: The filepath to the .csv containing the predictions
model_path: The filepath to the .pth containing the trained model

hilbert_dim: The input size dimension for the Hilbert curve function (must be a power of 2)
n_dim: The number of dimensions for the input
n_bits: The number of times the pattern repeats along a dimension

n_components: The number of PCA components to use for PCA
"""
### Main variables
model_type = "DenseNet121"
dim = 224
enhanced = False
row_idxs = []

### Only use csv_path and/or model_path if the file paths are not standard
csv_path = None
model_path = None

### Hilbert curve variables: These define the shape of the path of the Hilbert curve
if dim > 256:
    hilbert_dim = 512
else:
    hilbert_dim = 256
n_dim = 2
n_bits = 8

### PCA Variables
n_components = 10

### Save path for printed grad-CAM images
if enhanced:
    save_path = f"gradcams/{model_type}/BASE{dim}/RAW_{model_type}_BASE{dim}_{row_index}.png"
else:
    save_path = f"gradcams/{model_type}/BASE{dim}/ENHANCED_{model_type}_BASE{dim}_{row_index}.png"

##### Wrap all variables into a kwargs dict
gradcam_kwargs = dict(
    modeltype=model_type, dim=dim, enhanced=enhanced, row_idxs=row_idxs,
    csv_path=csv_path, model_path=model_path,
    hilbert_dim=hilbert_dim, n_dim=n_dim, n_bits=n_bits,
    n_components=n_components,
    save_path=save_path,
)

In [16]:
### Load and instantiate variables

### If not specified, build the csv and model file paths
if not gradcam_kwargs[csv_path]:
    if enhanced:
        csv_path = f"results/{model_type}_BASE{dim}_ENHANCED_120000samples_test_withPredictionsAndConfidences.csv"
    else:
        csv_path = f"results/{model_type}_BASE{dim}_120000samples_test_withPredictionsAndConfidences.csv"
if not gradcam_kwargs[model_path]:
    if enhanced:
        model_path = f"models/{model_type}_BASE{dim}_ENHANCED_120000samples.pth" # << Change this to your actual model path
    else:
        model_path = f"models/{model_type}_BASE{dim}_120000samples.pth" # << Change this to your actual model path

### Load the CSV file
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    raise ValueError(f"CSV file not found at: {csv_path}")
if df.empty:
    raise ValueError("The CSV file is empty!")

### Extract conditions (excluding metadata columns)
metadata_cols = ["Path", "Sex", "Age", "Frontal/Lateral", "AP/PA", 
                 "floating_file_path", "source_file_path", "base224_file_path",
                 "base224_file_path2", "base384_file_path", "base384_file_path2",
                 "base512_file_path", "base512_file_path2"]
conditions = [col for col in df.columns if col not in metadata_cols and not col.endswith("_pred") and not col.endswith("_confidence")]
n_classes = len(conditions)

### Instantiate the gradcam analysis variables
"""
Similarity/Distance is measured between gradcams of different conditions on the SAME x-ray
The variable is then averaged across the set
Use dot product similarity and L2 Distance
"""
gradcam_vars = dict(
    avg_dot = np.zeros((n_classes,n_classes)),
    avg_L2 = np.zeros((n_classes,n_classes)),
    avg_corr = np.zeros((n_classes,n_classes)),
    avg_explained_variance = 0,
    avg_explained_variance_ratios = np.zeros(n_components)
)

### Instantiate the Hilbert Curve index
hilbert_idxs = get_hilbert_index(hilbert_dim, n_dim, n_bits)

### Add new variables to gradcam+kwargs
gradcam_kwargs["n_classes"] = n_classes
gradcam_kwargs["conditions"] = conditions
gradcam_kwargs["hilbert_idxs"] = hilbert_idxs


In [ ]:
# -------------------------------
# Define Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Load Model from Checkpoint
# -------------------------------    
if model_type=="ResNet50":
    model = Custom_ResNet50(n_classes)
elif model_type=="DenseNet121":
    model = Custom_DenseNet121(n_classes)
else:
    raise ValueError("Invalid model_type input")
    
model.load_state_dict(torch.load(model_path, weights_only=True, map_location=device))  # Load weights
model.to(device)
model.eval()  # Set to evaluation mode